# T26 / E07 — Chunk-aware trên ngữ cảnh dài

E03 cho thấy chia theo đoạn có ích trên ViHallu, nơi ngữ cảnh chỉ có **5,3 đoạn**. E06 cho thấy
chú ý rơi đúng đoạn bằng chứng trên ISE-DSC01, nơi ngữ cảnh có **22,6 đoạn**.

Câu hỏi còn lại: khi ngữ cảnh dài gấp bốn, bộ phát hiện chunk-aware **có mạnh lên không**. Nếu
đóng góp của đề tài là "chia theo đơn vị nghĩa" thì đây phải là chỗ nó phát huy, không phải chỗ
nó đuối.

## Lượt này rẻ hơn vẻ ngoài: tập dev đã xong

E07 khai **cùng bộ dữ liệu, cùng cách chia đoạn, cùng bộ trích** với E06, nên hai bên dùng chung
một lượt trích (hash `15ef31521fd6`). Tập dev 3.646 mẫu đã trích ở T25 **không phải chạy lại**.

Còn phải trích:

| Tập | Số mẫu | Thời gian |
|---|---|---|
| train | 29.077 | ~8,7 giờ |
| test | 3.646 | ~1,1 giờ |
| dev | — | đã xong ở T25 |

## Vì sao trích đủ 29.077 chứ không lấy mẫu con

Đo đường cong học trên E03 bằng **chính đường ống này**, chạy CPU:

```
  1.400 mẫu train   0,7630
  2.800 mẫu train   0,7671
  4.200 mẫu train   0,7710
  5.600 mẫu train   0,7768
```

Gấp bốn dữ liệu mua được +0,0138 — nhỏ, nhưng vẫn dương và vẫn chưa bão hòa. Ba lý do khiến trích
đủ đáng làm:

1. **E16 (chuyển giao chéo bộ) dùng lại chính lượt trích này.** Trích thiếu bây giờ thì E16 thừa
   hưởng cùng giới hạn.
2. **Phép so khớp cỡ với E03 vẫn miễn phí** — ô 8 huấn luyện lại trên đúng 5.600 mẫu từ cùng
   shard, không tốn giây GPU nào. Trích đủ chỉ **thêm** một con số chứ không mất con số nào.
3. **Không phải xếp lịch GPU lần hai.**

## Đọc kết quả thế nào — viết trước khi thấy số

Hai con số, và chúng trả lời hai câu khác nhau:

- **Ô 7, huấn luyện trên đủ 29.077 mẫu** → E07 mạnh nhất có thể. Đây là số vào bảng kết quả.
- **Ô 8, huấn luyện trên đúng 5.600 mẫu** → so **có kiểm soát** với E03 (0,7567 trên test
  ViHallu). Chỉ con số này mới nói được "ngữ cảnh dài thì chunk-aware thế nào", vì nó giữ cỡ tập
  huấn luyện cố định.

Ba khả năng, viết ra trước để sau không đọc trại:

- **Số khớp cỡ cao hơn E03 rõ** → chunk-aware **mạnh lên** khi ngữ cảnh dài. Đóng góp của đề tài
  được củng cố ở đúng chỗ nó nên mạnh.
- **Xấp xỉ nhau** → phương pháp **bền** qua độ dài ngữ cảnh. Kết quả tốt, chỉ khiêm tốn hơn.
- **Thấp hơn rõ** → chunk-aware **không** chuyển được sang ngữ cảnh dài, và phải báo cáo đúng như
  vậy. Lúc đó E06 vẫn đứng (chú ý vẫn rơi đúng đoạn) nhưng khoảng cách giữa "định vị đúng" và
  "phân loại đúng" mới là thứ phải đi giải thích.

**Đừng so trực tiếp số của ô 7 với 0,7567 của E03.** Hai bộ dữ liệu khác nhau, cỡ train khác
nhau, độ khó khác nhau — đó là phép so không kiểm soát được biến nào.

## Chuẩn bị

Hai ô đầu giống notebook T25. Khoảng 2 phút.

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: 10c9e95 T26: công cụ E07 chunk-aware trên ngữ cảnh dài, chờ chạy GPU (#62)


In [2]:
# Ô 2 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit.
# hình đọc 7B. Không có nó thì mô hình phải nạp ở float16 và tràn 16 GB.
!pip install -q --no-deps -e .
!pip install -q -U transformers accelerate bitsandbytes

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.
vihallulens 0.1.0 requires rank-bm25, which is not install

In [3]:
# Ô 3 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, chạy CPU.
get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python scripts/normalize_data.py --dataset isedsc01")
get_ipython().system("python scripts/split_data.py --only isedsc01")
get_ipython().system("python -m pytest tests/test_chunking_config.py tests/test_assemble.py -q")


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : 10c9e95 T26: công cụ E07 chunk-aware trên ngữ cảnh dài, chờ chạy GPU (#62)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.16.1
  bitsandbytes     : 0.50.2
  accelerate       : 1.14.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA ISEDSC01
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  số dòng               : 36,369
  ngữ 

## Trích đặc trưng — khoảng 9,8 giờ

Tách tập train thành ba ô để có điểm dừng nhìn thấy được. **Mỗi ô chạy lại được**: mẫu nào tính
xong ghi xuống ngay, chạy lại thì bỏ qua phần đã có. Nên nếu phiên Kaggle chết giữa chừng, chỉ
cần mở phiên mới và chạy lại đúng ô đó — không mất phần đã trả tiền.

`--limit N` lấy N dòng đầu, và vì phần đã xong bị bỏ qua nên ba ô nối nhau thành ba chặng
0–10.000, 10.000–20.000, 20.000–29.077.

**Phiên Kaggle giới hạn 12 giờ.** 9,8 giờ vừa đủ nhưng sát; chia hai phiên cũng được, ô nào chưa
xong thì chạy lại ở phiên sau.

**Đọc gì trong lúc chạy:** cột `lỗi` phải là 0, và dòng `mẫu có bằng chứng` phải nói
`→ ghi thêm gold_rank cho E06`. Dòng `đã có sẵn` cho biết ô này nối tiếp đúng chỗ ô trước dừng.

In [4]:
# Ô 4 — trích tập train, tới mẫu 10.000. Khoảng 3.0 giờ.
!python scripts/extract_features.py --config configs/e07_chunk_aware_isedsc01.yaml \
    --split train --limit 10000


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e07_chunk_aware_isedsc01.yaml  (hash 15ef31521fd6)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : isedsc01/train, 10,000 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 9,997/10,000  → ghi thêm gold_rank cho E06
  đã có sẵn             : 0 mẫu trong isedsc01_train_15ef31521fd6.jsonl
  còn phải chạy         : 10,000 mẫu
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 3.39MB/s]
tokenizer_config.json: 7.30kB [00:00, 27.7MB/s]
vocab.json: 2.78MB [00:00, 111MB/s]
merges.txt: 1.67MB [00:00, 105MB/s]
tokenizer.json: 7.03MB [00:00, 144MB/s]
model.safetensors.index.json: 27.8kB [00:00, 85.6MB/s]
Fetching 4 files: 100%|█████████████

In [5]:
# Ô 5 — trích tập train, tới mẫu 20.000. Khoảng 3.0 giờ.
!python scripts/extract_features.py --config configs/e07_chunk_aware_isedsc01.yaml \
    --split train --limit 20000


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e07_chunk_aware_isedsc01.yaml  (hash 15ef31521fd6)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : isedsc01/train, 20,000 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 10,190/20,000  → ghi thêm gold_rank cho E06
  đã có sẵn             : 10,000 mẫu trong isedsc01_train_15ef31521fd6.jsonl
  còn phải chạy         : 10,000 mẫu
Loading weights: 100%|████████████████████████| 339/339 [00:16<00:00, 20.53it/s]


--------------------------------------------------------------------------------
  đã ghi thêm           : 10,000 mẫu, tổng 20,000
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/10,000
  có lớp tràn số        : 0/10,000
  thời gian   

In [6]:
# Ô 6 — trích tập train, phần còn lại, tới 29.077. Khoảng 2.7 giờ.
!python scripts/extract_features.py --config configs/e07_chunk_aware_isedsc01.yaml --split train


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e07_chunk_aware_isedsc01.yaml  (hash 15ef31521fd6)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : isedsc01/train, 29,077 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 19,046/29,077  → ghi thêm gold_rank cho E06
  đã có sẵn             : 20,000 mẫu trong isedsc01_train_15ef31521fd6.jsonl
  còn phải chạy         : 9,077 mẫu
Loading weights: 100%|████████████████████████| 339/339 [00:14<00:00, 23.85it/s]


--------------------------------------------------------------------------------
  đã ghi thêm           : 9,077 mẫu, tổng 29,077
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/9,077
  có lớp tràn số        : 0/9,077
  thời gian       

In [7]:
# Ô 7 — trích tập test. Khoảng 1,1 giờ.
!python scripts/extract_features.py --config configs/e07_chunk_aware_isedsc01.yaml --split test


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e07_chunk_aware_isedsc01.yaml  (hash 15ef31521fd6)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : isedsc01/test, 3,646 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 2,360/3,646  → ghi thêm gold_rank cho E06
  đã có sẵn             : 0 mẫu trong isedsc01_test_15ef31521fd6.jsonl
  còn phải chạy         : 3,646 mẫu
Loading weights: 100%|████████████████████████| 339/339 [00:14<00:00, 23.88it/s]


--------------------------------------------------------------------------------
  đã ghi thêm           : 3,646 mẫu, tổng 3,646
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/3,646
  có lớp tràn số        : 0/3,646
  thời gian             : 70.

## Chấm điểm

Chạy CPU. Ô 7 mất vài phút vì tập train lớn; ô 8 nhanh hơn.

In [8]:
# Ô 8 — chấm E07 trên đủ 29.077 mẫu train. Đây là số vào bảng kết quả.
!python scripts/run_chunk_aware.py --config configs/e07_chunk_aware_isedsc01.yaml


Thiếu đặc trưng của tập dev: data/processed/isedsc01_dev_15ef31521fd6.jsonl
Chạy trước: python scripts/extract_features.py --config configs/e07_chunk_aware_isedsc01.yaml --split dev


In [9]:
# Ô 9 — chấm lại trên đúng 5.600 mẫu train, khớp cỡ với E03.
# Đây mới là phép so có kiểm soát: cùng cỡ tập huấn luyện, chỉ khác độ dài ngữ cảnh.
!python scripts/run_chunk_aware.py --config configs/e07_chunk_aware_isedsc01.yaml \
    --train-sample 5600


Thiếu đặc trưng của tập dev: data/processed/isedsc01_dev_15ef31521fd6.jsonl
Chạy trước: python scripts/extract_features.py --config configs/e07_chunk_aware_isedsc01.yaml --split dev


In [10]:
# Ô 10 — kiểm toàn vẹn shard trước khi rời phiên. Vài chục giây, CPU.
import json
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config

run = extraction_hash(load_config("configs/e07_chunk_aware_isedsc01.yaml"))
expected = {"train": 29077, "dev": 3646, "test": 3646}
ok = True
for split, want in expected.items():
    path = Path("data/processed") / f"isedsc01_{split}_{run}.jsonl"
    if not path.exists():
        print(f"  THIEU {path.name}")
        ok = False
        continue
    rows = [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]
    ids = {r["sample_id"] for r in rows}
    blocks = [k for k in rows[0] if k.startswith(("lookback_", "chunk_", "top1_"))]
    good = len(rows) == want and len(ids) == want and len(blocks) == 7
    ok &= good
    print(f"  {'OK ' if good else 'HONG'} {path.name:<44} {len(rows):>6}/{want} dong, "
          f"{len(ids):>6} id, {len(blocks)} khoi")
print("\nBa shard hop le." if ok else "\nCO VAN DE - chay lai o trich truoc khi roi phien.")

  OK  isedsc01_train_15ef31521fd6.jsonl             29077/29077 dong,  29077 id, 7 khoi
  THIEU isedsc01_dev_15ef31521fd6.jsonl
  OK  isedsc01_test_15ef31521fd6.jsonl               3646/3646 dong,   3646 id, 7 khoi

CO VAN DE - chay lai o trich truoc khi roi phien.


In [11]:
# Ô 11 — lấy kết quả về. Shard train khoảng 1,4 GB.
import shutil
from pathlib import Path

shutil.copy("results/runs.jsonl", "/kaggle/working/runs.jsonl")
print("runs.jsonl")
for path in sorted(Path("data/processed").glob("isedsc01_*.jsonl")):
    shutil.copy(path, f"/kaggle/working/{path.name}")
    print(f"{path.name}  {path.stat().st_size / 1024**2:,.0f} MB")

runs.jsonl
isedsc01_test_15ef31521fd6.jsonl  189 MB
isedsc01_train_15ef31521fd6.jsonl  1,509 MB


## Sau khi chạy

Dán output của ô 7 và ô 8. Cần cả bảng chọn cách gộp đầu, khối `KẾT QUẢ TRÊN TẬP TEST`, và bảng
trọng số theo khối đặc trưng — khối cuối nói chunk-aware có thật sự được bộ phân loại dựa vào hay
không, và trên ngữ cảnh 22,6 đoạn thì câu trả lời đáng chú ý hơn hẳn trên 5,3 đoạn.

Rồi Quick Save, tải notebook về, chép đè lên `notebooks/t26_chunk_aware_ngu_canh_dai_t4.ipynb`.
**Đừng dùng Save & Run All** — nó chạy lại gần 10 giờ GPU.

Tải cả `runs.jsonl` và ba shard `isedsc01_*.jsonl` về (khoảng 1,9 GB tổng). **E16 dùng lại toàn
bộ**, nên tải một lần là xong.